# YOLO11 apmokymas naktiniais vaizdais (Google Colab)

Ši užrašų knygutė išmoko YOLO11 modelį atpažinti automobilius ir jų žibintus **naktį**, tada eksportuoja jį į `best.onnx`, kurį galima paleisti `ahb` prototipe be PyTorch.

**Prieš pradedant:** meniu *Runtime → Change runtime type → T4 GPU*. Be GPU mokymas užtruktų daugybę valandų.

Duomenys: Roboflow Universe rinkinys **„HeadLight Night“** (`road-mqx6z/headlight-night`, licencija CC BY 4.0). Klasės: naktinis automobilis, žibintas, tolimosios, artimosios, naktinis sunkvežimis. Jam atsisiųsti reikia nemokamo Roboflow API rakto.

In [ ]:
!pip install -q ultralytics roboflow
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Duomenų atsisiuntimas

1. Užsiregistruokite <https://roboflow.com> (nemokamai).
2. *Settings → API Keys* nukopijuokite **Private API Key** ir įrašykite žemiau.

In [ ]:
API_KEY = ""   # <- čia įklijuokite savo Roboflow API raktą

from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)

def parsisiusti(workspace, projektas, fmt="yolov8"):
    p = rf.workspace(workspace).project(projektas)
    try:
        versijos = p.versions()
        print(projektas, "versijos:", [str(v.version) for v in versijos])
    except Exception as e:
        print("Nepavyko gauti versijų sąrašo:", e)
        versijos = []
    for v in sorted(versijos, key=lambda v: int(str(v.version).split("/")[-1]), reverse=True):
        try:
            return v.download(fmt)
        except Exception as e:
            print("Versija", v.version, "– klaida:", e)
    return None

ds = parsisiusti("road-mqx6z", "headlight-night")
DIDELIS = False
if ds is None:
    print("\nHeadLight Night nepavyko – imu atsarginį rinkinį (34 037 vaizdai, gali užtrukti kelias minutes)...")
    ds = rf.workspace("duong-tran").project("vehicle-detection-night-time").version(5).download("yolov8")
    DIDELIS = True
print("Atsisiųsta į:", ds.location)


In [ ]:
import os, glob, yaml
root = ds.location
cfg = yaml.safe_load(open(f"{root}/data.yaml"))
print("Klasės:", cfg["names"])
splits = {s: len(glob.glob(f"{root}/{s}/images/*")) for s in ("train", "valid", "test")}
print("Vaizdų:", splits)

# keliai data.yaml faile – absoliutūs, kad ultralytics juos tikrai rastų
cfg["path"] = root
cfg["train"] = "train/images"
cfg["val"] = "valid/images" if splits["valid"] else "train/images"
cfg["test"] = "test/images" if splits["test"] else cfg["val"]
yaml.safe_dump(cfg, open(f"{root}/data_abs.yaml", "w"))
DATA = f"{root}/data_abs.yaml"

## 2. Mokymas

Perkeliamasis mokymas nuo COCO iš anksto išmokyto `yolo11n.pt`. Horizontalus apvertimas išjungtas (`fliplr=0`), nes eismo pusė turi reikšmės. Su T4 GPU ~60 epochų mažam rinkiniui užtrunka ~10–30 min.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data=DATA, epochs=15 if DIDELIS else 60, fraction=0.3 if DIDELIS else 1.0,
            imgsz=640, batch=16, fliplr=0.0, hsv_v=0.2,
            patience=15, project="runs", name="night", exist_ok=True)

In [ ]:
best = YOLO("runs/night/weights/best.pt")
m = best.val(data=DATA, split="test")
print(f"mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")

## 3. Eksportas į ONNX ir atsisiuntimas

`opset=12` ir fiksuotas 640 dydis – kad modelį galėtų paleisti OpenCV DNN prototipe.

In [ ]:
import shutil
path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(path, "best.onnx")
print("Klasės modelyje:", best.names)
from google.colab import files
files.download("best.onnx")

## 4. Naudojimas kompiuteryje

1. Atsisiųstą `best.onnx` įdėkite į `ahb` aplanką.
2. Vertinimas tikrais įrašais (visos šio modelio klasės reiškia automobilį ar jo žibintus, todėl `--yolo-classes all`):

```
py evaluate_real.py --video real/night.mp4  --labels real/night_cars.json  --model best.onnx --yolo-classes all --tag yolo_night
py evaluate_real.py --video real/vingis.mp4 --labels real/vingis_cars.json --model best.onnx --yolo-classes all --tag yolo_night
py evaluate_real.py --video real/naktis.mp4 --labels real/naktis_cars.json --model best.onnx --yolo-classes all --tag yolo_night
```

3. Peržiūra: `py main.py --source real/vingis.mp4 --yolo best.onnx --yolo-classes all`

## (Nebūtina) Didesnis rinkinys

Jei „HeadLight Night“ per mažas, galima naudoti **„Vehicle Detection Night time“** (`duong-tran/vehicle-detection-night-time`, 34 037 vaizdai). Jis didelis, todėl mokykite mažiau epochų (pvz., 15) ir su `fraction=0.3`. Pakeiskite 1 žingsnio langelyje workspace ir project pavadinimus ir paleiskite iš naujo.